In [ ]:
!pip install chromadb
!pip install sentence-transformers
!pip install groq
!pip install pypdf
!pip install numpy
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.6 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq
from pypdf import PdfReader
import numpy as np

# ---------------------------
# GROQ API
# ---------------------------
client = Groq(
    api_key="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"
)

# ---------------------------
# Read PDF Resumes
# ---------------------------
resume_files = [
    "/content/drive/MyDrive/akshaya /akshaya _resume.pdf",
    "/content/drive/MyDrive/akshaya /Abinaya_resume (1).pdf",
    "/content/drive/MyDrive/akshaya /Pooja_resume1.pdf"
]

documents = []

for file in resume_files:
    reader = PdfReader(file)

    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"

    documents.append({
        "filename": file,
        "content": text
    })

# ---------------------------
# Embedding Model
# ---------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [doc["content"] for doc in documents]

doc_embeddings = model.encode(texts).astype("float32")

# ---------------------------
# Create FAISS Index
# ---------------------------
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# ---------------------------
# Job Requirement Query
# ---------------------------
query = """
Looking for Python Developer with:
- Python
- Machine Learning
- Deep Learning
- SQL
- Data Analysis
- AI Projects
"""

query_embedding = model.encode([query]).astype("float32")

# ---------------------------
# Search Top Resumes
# ---------------------------
top_k = 3

distances, indices = index.search(
    query_embedding,
    top_k
)

print("\nTop Matching Resumes:\n")

retrieved_resumes = []

for rank, idx in enumerate(indices[0]):
    print(
        f"Rank {rank+1}: "
        f"{documents[idx]['filename']}"
    )

    retrieved_resumes.append(
        f"Resume Name: {documents[idx]['filename']}\n"
        f"{documents[idx]['content'][:4000]}"
    )

# ---------------------------
# Build Context
# ---------------------------
context = "\n\n".join(retrieved_resumes)

prompt = f"""
You are an expert HR recruiter.

Analyze the resumes based on the job requirement.

Job Requirement:
{query}

Resumes:
{context}

Tasks:
1. Rank candidates.
2. Give matching percentage.
3. List strengths.
4. List missing skills.
5. Recommend the best candidate.
6. Explain why.

Return in a clear table.
"""

# ---------------------------
# LLM Analysis
# ---------------------------
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

print("\n\n===== FINAL ANALYSIS =====\n")
print(response.choices[0].message.content)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Top Matching Resumes:

Rank 1: /content/drive/MyDrive/akshaya /akshaya _resume.pdf
Rank 2: /content/drive/MyDrive/akshaya /Pooja_resume1.pdf
Rank 3: /content/drive/MyDrive/akshaya /Abinaya_resume (1).pdf


===== FINAL ANALYSIS =====

**Candidate Analysis**

| Candidate | Rank | Matching Percentage | Strengths | Missing Skills |
| --- | --- | --- | --- | --- |
| Akshaya B | 1 | 90% | Python, Machine Learning, Deep Learning (not explicitly mentioned but has AI & Data Science background), SQL, Data Analysis, AI Projects (Udamey course recommendation system) | Deep Learning (not explicitly mentioned) |
| Pooja S | 2 | 80% | Python, Machine Learning, SQL, Data Analysis, AI Projects (Spam Mail Detection) | Deep Learning, Data Science (not explicitly mentioned) |
| Abinaya S | 3 | 70% | Python, Machine Learning, SQL, Data Analysis | Deep Learning, AI Projects (no relevant projects mentioned) |

**Recommendation:**
The best candidate for the Python Developer position is **Akshaya B**.

**Reas